# A4 — Multiple-Comparisons Correction (Benjamini–Hochberg FDR)

**Reviewer concern addressed:** Section A4 of the revision checklist — *"Many bin × metric combinations are tested without FDR or Bonferroni correction. Apply Benjamini–Hochberg FDR and note which findings survive."*

This notebook:
1. Re-runs every primary Mann–Whitney U test from RQ1 through RQ5 from scratch (reproducible from raw data).
2. Collects all raw p-values in a single table.
3. Applies Benjamini–Hochberg FDR correction via `statsmodels.stats.multitest.multipletests`.
4. Reports which findings survive correction.
5. Saves the full summary table to `revision_outputs/fdr_correction_summary.csv`.

In [ ]:
import pandas as pd
import numpy as np
import re
import ast
import os
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset', 'data', 'updated_dataset_metrics.csv')
OUT_DIR = os.path.join(os.getcwd(), 'revision_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw dataset: {df_raw.shape}")

# ── Numeric coercion ──────────────────────────────────────────────────────────
numeric_cols = [
    'doc_lines', 'doc_entropy', 'doc_readability', 'doc_code_overlap',
    'doc_redundancy', 'cyclomatic_complexity', 'sloc', 'semgrep_findings_count',
    'num_parameters', 'turnover_c5', 'turnover_c10', 'turnover_c20',
    'turnover_m1', 'turnover_m3'
]
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# ── Standard documented-function dataset (used by RQ1-RQ3, RQ5) ───────────────
df = df_raw.dropna(subset=['doc_entropy', 'doc_code_overlap', 'doc_redundancy']).copy()
df = df[df['doc_lines'] > 0].copy()
print(f"Documented-function dataset: {df.shape}")

# Convenience: tokenizer used across RQ1
def tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text.lower())

df['doc_token_count'] = df['doc_text'].apply(lambda x: len(tokenize(x)))

: 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper: run a two-sided Mann–Whitney U test and compute rank-biserial r
# ─────────────────────────────────────────────────────────────────────────────
def mw_test(a, b):
    """Return (stat, p, rank_biserial_r) for two-sided Mann-Whitney U."""
    a = np.asarray(a.dropna()) if hasattr(a, 'dropna') else np.asarray(a)
    b = np.asarray(b.dropna()) if hasattr(b, 'dropna') else np.asarray(b)
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    r = 1 - (2 * stat) / (len(a) * len(b))
    return stat, p, r

records = []  # will hold (rq, test_name, stat, p, r)

def record(rq, name, a, b):
    stat, p, r = mw_test(a, b)
    records.append({'RQ': rq, 'test': name, 'U_stat': stat, 'p_raw': p, 'rank_biserial_r': r})
    print(f"[{rq}] {name}: U={stat:.0f}, p={p:.3e}, r={r:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RQ1 tests: doc_code_overlap, doc_redundancy, doc_entropy, doc_token_count,
#            TTR, MTLD
# ─────────────────────────────────────────────────────────────────────────────
print('=== RQ1 ===')
ag = df[df['group'] == 'agent']
hu = df[df['group'] == 'human']

for m in ['doc_code_overlap', 'doc_redundancy', 'doc_entropy', 'doc_token_count']:
    record('RQ1', m, ag[m], hu[m])

# TTR and MTLD — require lexicalrichness
try:
    from lexicalrichness import LexicalRichness

    def get_lex(text):
        if not isinstance(text, str) or not text.strip():
            return np.nan, np.nan
        try:
            lex = LexicalRichness(text)
            return lex.ttr, lex.mtld(threshold=0.72)
        except Exception:
            return np.nan, np.nan

    df_lex = df[df['doc_text'].notna() & (df['doc_text'].str.strip() != '')].copy()
    lex_res = df_lex['doc_text'].apply(get_lex)
    df_lex[['ttr', 'mtld']] = pd.DataFrame(lex_res.tolist(), index=df_lex.index)

    ag_l = df_lex[df_lex['group'] == 'agent']
    hu_l = df_lex[df_lex['group'] == 'human']
    record('RQ1', 'ttr', ag_l['ttr'], hu_l['ttr'])
    record('RQ1', 'mtld', ag_l['mtld'], hu_l['mtld'])
except ImportError:
    print('lexicalrichness not installed — skipping TTR/MTLD')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RQ2 tests: code quality metrics + bin-level doc_token comparisons
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== RQ2 ===')

df2 = df.copy()
df2['doc_tokens'] = df2['doc_text'].apply(lambda x: len(tokenize(x)))

# Agent vs developer on code quality metrics
for m in ['cyclomatic_complexity', 'sloc', 'semgrep_findings_count']:
    record('RQ2', f'code_quality_{m}', df2[df2['group']=='agent'][m], df2[df2['group']=='human'][m])

# Bin-level tests: doc_tokens within each CC bin
cc_bins   = [0, 2, 10, float('inf')]
cc_labels = ['Simple (1-2)', 'Moderate (3-10)', 'Complex (11+)']
df2['cc_bin'] = pd.cut(df2['cyclomatic_complexity'], bins=cc_bins, labels=cc_labels)

for b in cc_labels:
    sub = df2[df2['cc_bin'] == b]
    a_vals = sub[sub['group']=='agent']['doc_tokens']
    h_vals = sub[sub['group']=='human']['doc_tokens']
    if len(a_vals) >= 5 and len(h_vals) >= 5:
        record('RQ2', f'doc_tokens|cc_bin={b}', a_vals, h_vals)
    else:
        print(f'  Skipping cc_bin={b}: too few observations (agent={len(a_vals)}, human={len(h_vals)})')

# SLOC bins
sloc_bins   = [0, 10, 50, float('inf')]
sloc_labels = ['Short (<10)', 'Medium (11-50)', 'Long (51+)']
df2['sloc_bin'] = pd.cut(df2['sloc'], bins=sloc_bins, labels=sloc_labels)

for b in sloc_labels:
    sub = df2[df2['sloc_bin'] == b]
    a_vals = sub[sub['group']=='agent']['doc_tokens']
    h_vals = sub[sub['group']=='human']['doc_tokens']
    if len(a_vals) >= 5 and len(h_vals) >= 5:
        record('RQ2', f'doc_tokens|sloc_bin={b}', a_vals, h_vals)
    else:
        print(f'  Skipping sloc_bin={b}: too few observations')

# Semgrep bins
sg_bins   = [-1, 0, 2, float('inf')]
sg_labels = ['Clean (0)', 'Low Issues (1-2)', 'High Issues (3+)']
df2['sg_bin'] = pd.cut(df2['semgrep_findings_count'], bins=sg_bins, labels=sg_labels)

for b in sg_labels:
    sub = df2[df2['sg_bin'] == b]
    a_vals = sub[sub['group']=='agent']['doc_tokens']
    h_vals = sub[sub['group']=='human']['doc_tokens']
    if len(a_vals) >= 5 and len(h_vals) >= 5:
        record('RQ2', f'doc_tokens|sg_bin={b}', a_vals, h_vals)
    else:
        print(f'  Skipping sg_bin={b}: too few observations (agent={len(a_vals)}, human={len(h_vals)})')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RQ3 tests: num_parameters bins
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== RQ3 ===')

df3 = df.copy()
df3['doc_tokens'] = df3['doc_text'].apply(lambda x: len(tokenize(x)))

param_bins   = [-1, 0, 2, 5, float('inf')]
param_labels = ['None (0)', 'Few (1-2)', 'Moderate (3-5)', 'Many (6+)']
df3['param_bin'] = pd.cut(df3['num_parameters'], bins=param_bins, labels=param_labels)

for b in param_labels:
    sub = df3[df3['param_bin'] == b]
    a_vals = sub[sub['group']=='agent']['doc_tokens']
    h_vals = sub[sub['group']=='human']['doc_tokens']
    if len(a_vals) >= 5 and len(h_vals) >= 5:
        record('RQ3', f'doc_tokens|param_bin={b}', a_vals, h_vals)
    else:
        print(f'  Skipping param_bin={b}: too few observations')

# Overall num_parameters distribution
record('RQ3', 'num_parameters_overall', df3[df3['group']=='agent']['num_parameters'], df3[df3['group']=='human']['num_parameters'])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RQ4 tests: turnover at each time window
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== RQ4 ===')

def extract_turnover(val):
    if pd.isna(val) or val == -1 or val == '-1':
        return np.nan
    try:
        if isinstance(val, str) and '(' in val:
            parsed = ast.literal_eval(val)
            val = parsed[1]
        num = float(val)
        return num if num >= 0 else np.nan
    except Exception:
        return np.nan

df4 = df_raw.dropna(subset=['doc_entropy', 'doc_code_overlap', 'doc_redundancy']).copy()
df4 = df4[df4['doc_lines'] > 0].copy()

turnover_cols = ['turnover_c5', 'turnover_c10', 'turnover_c20', 'turnover_m1', 'turnover_m3']
for col in turnover_cols:
    df4[col] = df4[col].apply(extract_turnover)

df4_turn = df4.dropna(subset=turnover_cols)
print(f"Turnover dataset: {df4_turn.shape}")

for col in turnover_cols:
    a_vals = df4_turn[df4_turn['group']=='agent'][col]
    h_vals = df4_turn[df4_turn['group']=='human'][col]
    record('RQ4', col, a_vals, h_vals)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RQ5 tests: sentiment (VADER) and readability
# ─────────────────────────────────────────────────────────────────────────────
print('\n=== RQ5 ===')

import re as _re

def clean_comments(text):
    if not isinstance(text, str):
        return ''
    blocks     = _re.findall(r'/\*+([\s\S]*?)\*/', text)
    docstrings = _re.findall(r'["\']{{3}}([\s\S]*?)["\']{{3}}', text)
    inline     = _re.findall(r'(?:#|//)\s*(.*?)\s*(?=#|//|$)', text)
    combined   = blocks + docstrings + inline
    parts = []
    for item in combined:
        clean = _re.sub(r'^\s*\* ?', '', item, flags=_re.MULTILINE)
        clean = ' '.join(clean.split())
        if clean.strip():
            parts.append(clean.strip())
    return ' '.join(parts)

df5 = df.copy()
df5['clean_text'] = df5['doc_text'].apply(clean_comments)

# VADER sentiment
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyzer = SentimentIntensityAnalyzer()
    vader_cols = ['neg', 'neu', 'pos', 'compound']
    vader_scores = df5['clean_text'].apply(lambda t: analyzer.polarity_scores(t) if isinstance(t, str) and t.strip() else {'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0})
    df5[vader_cols] = pd.DataFrame(vader_scores.tolist(), index=df5.index)
    df5['author_type'] = df5['group'].map({'agent': 'Agentic', 'human': 'Developer'})

    for m in vader_cols:
        a_vals = df5[df5['author_type']=='Agentic'][m]
        h_vals = df5[df5['author_type']=='Developer'][m]
        record('RQ5', f'vader_{m}', a_vals, h_vals)
except ImportError:
    print('vaderSentiment not installed — skipping VADER tests')

# Readability
try:
    import textstat
    read_metrics = ['flesch_reading_ease', 'smog_index', 'flesch_kincaid_grade']

    def score_read(text):
        if not isinstance(text, str) or not text.strip():
            return {m: np.nan for m in read_metrics}
        return {
            'flesch_reading_ease': textstat.flesch_reading_ease(text),
            'smog_index': textstat.smog_index(text),
            'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text)
        }

    read_scores = df5['clean_text'].apply(score_read)
    df5[read_metrics] = pd.DataFrame(read_scores.tolist(), index=df5.index)

    for m in read_metrics:
        a_vals = df5[df5['group']=='agent'][m].dropna()
        h_vals = df5[df5['group']=='human'][m].dropna()
        record('RQ5', f'readability_{m}', a_vals, h_vals)
except ImportError:
    print('textstat not installed — skipping readability tests')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Apply Benjamini–Hochberg FDR correction
# ─────────────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(records)
print(f"\nTotal tests collected: {len(results_df)}")

reject, p_corrected, _, _ = multipletests(results_df['p_raw'], method='fdr_bh', alpha=0.05)

results_df['p_fdr_bh']            = p_corrected
results_df['significant_raw']     = results_df['p_raw'] < 0.05
results_df['significant_fdr_bh']  = reject

# Format for display
display_df = results_df[['RQ', 'test', 'p_raw', 'p_fdr_bh', 'significant_raw', 'significant_fdr_bh', 'rank_biserial_r']].copy()
display_df['p_raw']    = display_df['p_raw'].map('{:.4e}'.format)
display_df['p_fdr_bh'] = display_df['p_fdr_bh'].map('{:.4e}'.format)
display_df['rank_biserial_r'] = display_df['rank_biserial_r'].map('{:.4f}'.format)

display(display_df)

## Which findings survive BH correction?

The table below summarises the outcome. Any test marked `significant_fdr_bh = True` remains statistically significant after controlling the false discovery rate at α = 0.05.

> **Key question for the paper:** Do the core claims (agents produce more overlapping/redundant documentation, agents produce more tokens in complex-CC functions) survive FDR correction?  
> Examine the `significant_fdr_bh` column above for each primary metric.  
> Tests that lose significance after correction should be framed as *exploratory* in the manuscript.

In [ ]:
# Summary: surviving vs not surviving
n_total    = len(results_df)
n_raw_sig  = results_df['significant_raw'].sum()
n_fdr_sig  = results_df['significant_fdr_bh'].sum()
n_lost     = n_raw_sig - n_fdr_sig

print(f"Total tests:               {n_total}")
print(f"Significant before FDR:    {n_raw_sig} / {n_total}")
print(f"Significant after BH-FDR:  {n_fdr_sig} / {n_total}")
print(f"Findings LOST after FDR:   {n_lost}")
print()

if n_lost > 0:
    print('Tests that lost significance after FDR correction:')
    lost = results_df[results_df['significant_raw'] & ~results_df['significant_fdr_bh']]
    for _, row in lost.iterrows():
        print(f"  [{row['RQ']}] {row['test']}  p_raw={row['p_raw']:.4e}  p_fdr={row['p_fdr_bh']:.4e}")
else:
    print('All raw-significant tests survive BH-FDR correction.')

# Save
out_path = os.path.join(OUT_DIR, 'fdr_correction_summary.csv')
results_df.to_csv(out_path, index=False)
print(f"\nSaved to {out_path}")

## Revision note

- **What this adds:** A single reproducible table covering every primary Mann–Whitney U test in the paper, with Benjamini–Hochberg FDR-adjusted p-values. Empirical SE venues now routinely expect this.
- **Where to cite in paper:** Add a paragraph to Section 3.5 (or Statistical Analysis subsection) describing the correction procedure, and update any specific p-value claims in Sections 4.1–4.5 and Table 2/3 to note that they survive FDR correction.
- **What to watch for:** If any bin-level test (especially the Semgrep *High Issues* bin, n=16) loses significance after correction, move it from a primary result to a *limitations/exploratory* note — this also addresses reviewer concern B3.